# 📸 debim: Full 3D Visual Regression & Component-Level Benchmark

[![GitHub Repository](https://img.shields.io/badge/GitHub-PRIDA--TAKON%2Fdebim-181717?logo=github&style=for-the-badge)](https://github.com/PRIDA-TAKON/debim)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg?style=for-the-badge)](https://github.com/PRIDA-TAKON/debim/blob/main/LICENSE)

> **debim** (Declarative BIM) Full 3D Component-Level Visual Regression Suite.
> Evaluates **100% of all components (no sampling)** across representative architectural and structural buildings.
>
> Prevents false-positive conversions where YAML is syntactically valid but 3D geometry is misoriented or displaced.

In [ ]:
# 1. Install dependencies & clone latest debim main branch
!pip install -q ifcopenshell "pydantic>=2.0" pyyaml rich trimesh matplotlib pillow pandas tabulate

!rm -rf /kaggle/working/debim
!git clone https://github.com/PRIDA-TAKON/debim.git /kaggle/working/debim

import sys
sys.path.insert(0, "/kaggle/working/debim/src")
sys.path.insert(0, "/kaggle/working/debim")
print("Environment initialized with latest debim commit from GitHub.")

In [ ]:
# 2. Discover and select curated diverse representative IFC models
from pathlib import Path

input_root = Path("/kaggle/input")
all_ifc_files = sorted(list(input_root.glob("**/*.ifc")))
print(f"Found {len(all_ifc_files)} IFC files across datasets.")

# Pick distinct benchmark buildings across categories
selected_models = []
seen_names = set()
for f in all_ifc_files:
    if f.stat().st_size > 15000 and f.stat().st_size < 15000000: # between 15KB and 15MB
        if f.name not in seen_names:
            seen_names.add(f.name)
            selected_models.append(f)
            if len(selected_models) >= 8:
                break

print(f"Selected {len(selected_models)} representative models for 100% exhaustive component testing:")
for idx, f in enumerate(selected_models, 1):
    print(f"  [{idx}] {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 3. Execute 100% Exhaustive 3D Visual Regression (Testing EVERY piece)
import os
from tools.render_visual_regression import run_visual_regression

output_dir = Path("/kaggle/working/visual_regression_output")
output_dir.mkdir(parents=True, exist_ok=True)

summary_stats = []

for idx, ifc_p in enumerate(selected_models, 1):
    print(f"\n[{idx}/{len(selected_models)}] Exhaustive 100% Component Test: {ifc_p.name}")
    model_out = output_dir / ifc_p.stem
    try:
        # max_elements_per_type=99999 and limit_total=99999 means NO SAMPLING: TEST EVERY PIECE
        stat = run_visual_regression(
            ifc_path=ifc_p,
            output_dir=model_out,
            max_elements_per_type=99999,
            limit_total=99999,
        )
        summary_stats.append({"name": ifc_p.name, **stat})
    except Exception as e:
        print(f"  [SKIP] Error processing {ifc_p.name}: {e}")

print("\n100% Exhaustive visual regression execution complete.")

In [ ]:
# 4. Display KPI Summary & Visual Diff Gallery
import pandas as pd
from IPython.display import display, HTML, Markdown, Image

if summary_stats:
    df = pd.DataFrame(summary_stats)
    total_tested = df["total_tested"].sum()
    total_passed = df["pass_count"].sum()
    overall_pass_rate = (total_passed / total_tested * 100) if total_tested else 0.0
    mean_match = df["avg_match"].mean()

    display(Markdown(f"""
### 📊 100% Exhaustive 3D Visual Regression KPIs
- **Total Buildings Evaluated:** {len(df)}
- **Total 3D Components Tested (100% Coverage):** {total_tested} pieces
- **Overall Component Pass Rate (>=85%):** {total_passed}/{total_tested} ({overall_pass_rate:.1f}%)
- **Mean Visual Fidelity Match:** {mean_match:.1f}%
"""))
    display(df[["name", "total_tested", "pass_count", "avg_match"]])

    # Gallery of sample diff images
    all_images = list(output_dir.glob("**/*.png"))
    print(f"\nGenerated {len(all_images)} 3D component diff images.")
    display(Markdown("### 🖼️ Sample 3D Visual Diff Gallery (Original | debim | Diff Map)"))
    for img_p in all_images[:12]:
        display(Markdown(f"**{img_p.parent.name} - {img_p.stem}**"))
        display(Image(filename=str(img_p)))
else:
    print("No models evaluated.")